# Fluxo de IA - NPS Inteligente

Este notebook reproduz o fluxo de inteligência artificial utilizado para avaliar atendimentos contábeis.

### Fases do Processo:
1. **Metadados**: Extração de dados estruturados (cliente, atendente, datas).
2. **Tagueamento**: Identificação do serviço principal e marcadores de erro.
3. **Montagem do Prompt**: Preparação do contexto para avaliação.
4. **Score das Dimensões**: Avaliação individual de Comunicação, Profissionalismo e Resolução.
5. **Consolidação**: Cálculo de médias e agregação de resultados.
6. **Resposta**: Geração do resumo final e oportunidades de melhoria.

In [2]:
import os
import json
import re
from datetime import datetime
from openai import OpenAI
from dotenv import load_dotenv

# Carregar variáveis de ambiente (subir um nível para achar o .env)
load_dotenv('../.env')

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

def call_openai(prompt, model=MODEL, json_mode=False):
    response_format = { "type": "json_object" } if json_mode else None
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        response_format=response_format
    )
    return response.choices[0].message.content

def load_prompt(filename):
    # Subir um nível para achar a pasta src/prompts
    path = f"../src/main/resources/prompts/{filename}"
    if not os.path.exists(path):
        # Fallback para .md se não for .txt
        path = path.replace('.txt', '.md') if '.txt' in path else path + '.md'
        
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

def clean_json(text):
    """Extrai JSON de blocos de código markdown se necessário"""
    match = re.search(r'\{.*\}', text, re.DOTALL)
    return match.group(0) if match else text

## Dados de Teste
Abaixo, um exemplo de chat para processamento.

In [3]:
SAMPLE_CHAT = """
Cliente: Olá, gostaria de saber o valor do meu FGTS que vence amanhã.
Atendente: Olá! Sou o João da Contabilidade ABC. Vou verificar para você.
Atendente: O valor é R$ 450,00. Segue a guia anexa.
Cliente: Mas eu achei que seria mais, no mês passado foi R$ 600,00.
Atendente: Ah, desculpe! Esqueci de considerar o funcionário novo que entrou dia 10. 
Atendente: O valor correto é R$ 680,00. Já estou gerando a guia retificada.
Cliente: Agora sim, obrigado João.
Atendente: De nada! Posso ajudar em algo mais?
Cliente: Não, só isso. Tchau.
"""

## 1. Metadados
Extração de dados estruturados do atendimento.

In [ ]:
print("--- Fase 1: Metadados ---")
prompt_meta = load_prompt("Extrair Metadados")
prompt_meta = prompt_meta.replace("{{input}}", SAMPLE_CHAT)

res_meta_raw = call_openai(prompt_meta, json_mode=True)
metadata = json.loads(clean_json(res_meta_raw))
print(json.dumps(metadata, indent=2, ensure_ascii=False))

--- Fase 1: Metadados ---


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-******************************************gYm-. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

: 

## 2. Tagueamento
Identificação de serviço e erros (marcadores).

In [ ]:
print("\n--- Fase 2: Tagueamento ---")
tag_prompt = """
Analise o atendimento abaixo e identifique:
1. Serviço principal (Ex: FGTS, Folha, Impostos, Admissão, etc. - Máximo 2 palavras)
2. Marcadores de erro (Lista de erros encontrados no formato #Erro, ex: #ErroCalculo #Esquecimento)

Retorne apenas um JSON:
{
  "servico_principal": "",
  "marcadores": []
}

Texto:
"""
res_tags_raw = call_openai(tag_prompt + SAMPLE_CHAT, json_mode=True)
tags = json.loads(clean_json(res_tags_raw))
print(json.dumps(tags, indent=2, ensure_ascii=False))

## 3 e 4. Score das Dimensões
Avaliação de cada pilar individualmente.

In [ ]:
dimensoes = [
    "Comunicação e Clareza",
    "Profissionalismo e Conformidade",
    "Resolução e Eficiência"
]

scores = []

for dim in dimensoes:
    print(f"\n--- Fase 4: Avaliando {dim} ---")
    prompt_dim = load_prompt(dim)
    prompt_dim = prompt_dim.replace("{{input}}", SAMPLE_CHAT)
    # Limpar placeholders opcionais
    prompt_dim = prompt_dim.replace("{{contexto_feedback}}", "")
    prompt_dim = prompt_dim.replace("{{contexto_global}}", "")
    
    res_dim_raw = call_openai(prompt_dim, json_mode=True)
    score_data = json.loads(clean_json(res_dim_raw))
    score_data["dimensao"] = dim
    scores.append(score_data)
    print(f"Nota: {score_data.get('nota')} - Justificativa: {score_data.get('justificativa')[:100]}...")

## 5. Consolidação
Agregação dos resultados.

In [ ]:
print("\n--- Fase 5: Consolidação ---")
notas = [s.get('nota', 0) for s in scores]
media = sum(notas) / len(notas)

resultado_consolidado = {
    "metadata": metadata,
    "tags": tags,
    "scores": scores,
    "media": round(media, 2)
}
print(f"Média Final: {resultado_consolidado['media']}")

## 6. Resposta Final
Geração do output formatado para o usuário.

In [ ]:
print("\n--- Fase 6: Resposta ---")
prompt_resumo = load_prompt("Resumir Atendimento")
prompt_resumo = prompt_resumo.replace("{{input}}", SAMPLE_CHAT)
resumo_texto = call_openai(prompt_resumo)

# Montar Resposta Final (Simulando o prompt_original.txt)
output = f"""
Atendimento: {metadata.get('numero_atendimento')}
Atendente: {metadata.get('atendente_principal')}
Cliente: {metadata.get('nome_cliente')}
Serviço principal: {tags.get('servico_principal')}
Marcadores: {' '.join(tags.get('marcadores', []))}

Nota FINAL: {resultado_consolidado['media']}
Resumo: {resumo_texto}

Notas:
"""
for s in scores:
    output += f"{s['dimensao']} - Nota: {s['nota']}: {s['justificativa']}\n"

print(output)